In [17]:
import os
from pathlib import Path
import certifi 
from dotenv import load_dotenv

from langchain_community.chat_models import ChatOllama
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.utilities.tavily_search import TavilySearchAPIWrapper
from langchain import hub
from langchain.tools import tool
import requests

In [18]:
from langchain.agents import create_react_agent, AgentExecutor

In [19]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()

search_dir = Path.cwd()
search_dirs = (search_dir, *search_dir.parents)
dotenv_path = None
for directory in search_dirs:
    project_root = directory if directory.name == "Langchain-Single-Agent" else directory / "Langchain-Single-Agent"
    project_env = project_root / ".env"
    if project_env.is_file():
        dotenv_path = project_env
        break

if dotenv_path is None:
    for directory in search_dirs:
        candidate = directory / ".env"
        if candidate.is_file():
            dotenv_path = candidate
            break

if dotenv_path is None:
    raise FileNotFoundError("Could not find .env. Create it in the Langchain-Single-Agent project folder.")

load_dotenv(dotenv_path=dotenv_path, override=True)
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [20]:
if TAVILY_API_KEY:
    search_tool = TavilySearchResults(
        api_wrapper=TavilySearchAPIWrapper(tavily_api_key=TAVILY_API_KEY),
        max_results=2
    )
else:
    @tool
    def search_tool(query: str) -> str:
        """Search the web for information."""
        return "Web search is unavailable because TAVILY_API_KEY is not configured."

In [21]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [22]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://www.crescendo.ai/news/latest-ai-news-and-updates',
  'content': 'Date: May 21, 2026\n\nSummary: SoundHound AI announced the acquisition of conversational AI platform LivePerson, combining its voice and audio recognition capabilities with LivePerson\'s digital messaging products. Management has set 2027 revenue targets of $350 million to $400 million and identified a $500 million cross-selling opportunity across the combined customer base. The deal positions SoundHound to compete more directly with Microsoft, Google, and Amazon across enterprise contact centers, healthcare, and financial services.\n\nSource: Simply Wall St ↗\n\n### (AI News) Nvidia\'s Memory Costs Soar 485% as Latest AI Systems Now Cost $7.8 Million to Build\n\nDate: May 21, 2026 [...] Source: 425 Business\n\n### (AI Innovation) Microsoft Launches $2.5 Billion Frontier Company for Enterprise AI Deployment\n\nDate: July 2, 2026\n\nSummary: Microsoft announced Microsoft Frontier Company, backed by $2.5 b

In [23]:
# ==========================================
# LOCAL OLLAMA LLM
# ==========================================

llm = ChatOllama(
    model="llama3.2:latest",
    temperature=0
)

In [24]:
response = llm.invoke("Tell me a joke about AI")
response 

AIMessage(content='Why did the AI program go to therapy?\n\nBecause it was struggling to process its emotions.', response_metadata={'model': 'llama3.2:latest', 'created_at': '2026-10-10T10:55:51.5905029Z', 'message': {'role': 'assistant', 'content': ''}, 'done': True, 'done_reason': 'stop', 'total_duration': 1361491500, 'load_duration': 34256900, 'prompt_eval_count': 31, 'prompt_eval_cached_count': 20, 'prompt_eval_duration': 220360000, 'eval_count': 19, 'eval_duration': 1087362000}, id='run-fb57c7c9-7fc4-44bb-90d2-ee784dbc3014-0')

In [25]:
# ==========================================
# PROMPT
# ==========================================

prompt = hub.pull("hwchase17/react")

c:\project\LLM_learning\Complete-Agentic-AI-Course\Langchain-Single-Agent\.venv311\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [26]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [27]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [28]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [ ]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [ ]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "Find the capital of India"
        "and then find its current weather."
    )
})


In [29]:
print(response["output"])

TypeError: 'AIMessage' object is not subscriptable